## Imports

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [4]:
reviews_preprocessed = pd.read_csv("../data/processed/reviews_preprocessed.csv", index_col=0)
reviews_preprocessed = reviews_preprocessed.dropna(subset=["Text_clean"])

X = reviews_preprocessed["Text_clean"]
y = reviews_preprocessed["Sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LinearSVC(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

In [5]:
results = pd.DataFrame({
    "Text": reviews_preprocessed.loc[X_test.index, "Text"],
    "True_Sentiment": y_test,
    "Predicted_Sentiment": y_pred
})

In [6]:
false_positives = results[(results["True_Sentiment"] == 0) & (results["Predicted_Sentiment"] == 1)]
false_negatives = results[(results["True_Sentiment"] == 1) & (results["Predicted_Sentiment"] == 0)]

print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

False positives: 3864
False negatives: 2097


In [7]:
pd.set_option('display.max_colwidth', 300)
false_positives[["Text"]].sample(10, random_state=42)

,Text
Id,
345369,"great coffee, have been very pleased. package however came with the middle one having rips and tears. coffee leaking everywhere. someone should have caught this?"
53420,"these are tasty powdered pecans. amazon fulfillment warehouses apparently have an even worse policy: zero padding in a box. to experience the ""quality"" of pecans i received place your pecans in a paint shaker for an hour."
223649,"one's first thoughts on eating this cereal are of the bathroom, and for two reasons. first, that's where its taste seems to come from, and second, because that's where any mouthful should immediately be coughed up. the best that can be said about krave is that it will, one hopes, soon be off the..."
429166,"i love ordering from amazon, but unfortunately i have nothing good to say about taste of the wild dog food. i have been feeding this food to my 3 dogs for several years now, but with the most recent dog food recall affecting totw, i am switching to a different brand. taste of the wild is made by..."
410357,"i was really excited to try these cookies because i really love oreos. after reading rave reviews, i went out and got a package of them from whole foods. maybe it's the fact that i have just recently been diagnosed as celiac (1 month ago) and the amazing taste of real oreos is fresh in my mind, ..."
19704,i wanted this for the omega 3's but my daughter wouldn't eat this. it really smells and looks like cat food.
270990,i read a review and was excited to try this salt. i didn't think it was anything special and would highly recommend morton's kosher salt which is far superior and more flavorable in my opinion. it is available at any grocery store and much cheaper too!
518573,i moved into a building that had roaches but was not told about this problem beforehand. i bought these prior to my apartment being sprayed for the third time. i don't know if my building has a special strain of finicky and pesticide resistant roaches. but nothing seems to work. this product was...
449499,"we have 2 tomcats and this is the first time i've tried the litter pearls and i hate them and the ""boys"" do too. they don't clump the waste in the litter tray and they don't cover it either, not like the other brands, so you see most of their poop and it smells unlike the other brands and it is ..."


### False positives: mixed-sentiment reviews

A recurring pattern in false positives (true negative, predicted positive) is reviews that
open with positive language about the product itself, followed by a negative turn (often
signaled by "however", "but", "unfortunately") — typically about packaging, shipping, or an
unrelated issue. Example: "great coffee, have been very pleased. package however came..."

This reveals a structural limitation of TF-IDF: it treats text as a bag of words with no
regard for word order or position. A review's overall sentiment often hinges on which part
carries more weight for the author (frequently the later part, after a contrastive conjunction),
but TF-IDF has no way to capture that — "great" early in the text and "damaged" later
contribute equally to the vector, regardless of which one actually drove the author's rating.

This is a known limitation that sequence-aware models (e.g. transformer-based architectures)
handle better, since they process text with attention to word order and context.

In [13]:
false_negatives[["Text"]].sample(10, random_state=42)

,Text
Id,
60300,"this product tastes like fresh milk but it needs to be well blended for the milk powder to get into solution. i use purified water, which is healthier than the increased herbicides and other volatile organic compounds in tap or bottled waters. organic valley is that this is a cooperative of smal..."
568232,"we're ordered the fruitables dog treats in the past and keep coming back for more. they're good for the dogs and our dogs love them, too. we've even given them to friends and family for their dogs."
219770,being gluten free has meant nasty cardboard blah crackers...until i found these. i am thrilled beyond words. they taste like the regular bagel chips. no one would be able to tell the difference. yeah!!!!
14877,this is a very good product.however due to poor packaging by nutricity two bottles got broken.this had to be discarded by the courier company and the remaining ten bottles were repacked by them.nutricity refuses to entertain any claim as they say that the time to make a claim has elasped.
487483,"not like some mediocre juice [especially apple], that's, sometimes, weak on flavor.......very robust natural grape. you have to add sugar; way to tart. you can dilute by 1/2 and it's still very good. makes it much more economical than it looks............ love it. p.s......it does contain some a..."
207085,"i have used this food for 3 months, and at first i thought it was great. but have since taken all my dogs off of it. their coats are dry and have bad hair loss. some dogs it is hard to keep weight on them with this food."
150255,"i bought this as a subscription for my daughter's cats, as our own cats like this. two of the three of my daughter's ones won't touch it, very strange, they are too fussy !"
194275,when my friend first showed me one i thought they were real rocks. then after trying one i realized not only do they look like rocks they taste 10 times better than any other chocolate i ever had. go chocolate rocks!!!!!!!!!!!
90323,"since the ingredients weren't printed on the amazon listing, i contacted the manufacturer and asked before purchasing. i received a prompt, courteous, accurate reply. the ingredients are vanilla extracts and alcohol. for comparison, the ingredients in the vanilla extract we purchased at the groc..."


### False negatives: text-rating inconsistency

Some false negatives (true positive, predicted negative) show the reverse pattern: text that
turns negative partway through, but where the author's final Score remained high. Example:
"i have used this food for 3 months, and at first i thought it was great. but have since taken
all my dogs off of it. their coats are dry and have bad hair loss..." — despite explicitly
negative language ("bad hair loss", "taken off of it"), this review was scored positively (4-5)
by its author.

Unlike the false positive pattern above (where the model missed a genuine negative turn that
matched the author's low score), this reflects a different issue: **inconsistency between the
review text and the star rating itself**, not a modeling limitation. The model reasonably
picked up on the negative language, but the ground truth label (derived from Score) didn't
match the sentiment expressed in that portion of the text. This kind of label noise is inherent
to real-world review data and sets a practical ceiling on achievable accuracy — no text-only
model can fully resolve cases where the author's own rating doesn't align with their written
sentiment.

### Project summary

This project built a binary sentiment classifier on ~520K Amazon Fine Food Reviews, covering
the full pipeline from raw text to a documented, evaluated model:

- **EDA**: identified class imbalance in Score, HTML artifacts in review text, and duplicate
  reviews requiring cleanup.
- **Preprocessing**: built a spaCy-based cleaning pipeline (tokenization, lemmatization, stop
  word removal), with two deliberate, documented decisions — preserving contractions by not
  stripping punctuation before tokenization, and excluding negation words ("not", "no") from
  the stop word list to avoid losing sentiment-critical information.
- **Modeling**: established a TF-IDF + Logistic Regression baseline (macro F1 0.85), then
  iteratively improved it through class weighting (a precision/recall trade-off, not a net
  improvement), bigram features (a genuine improvement to macro F1 0.88), and algorithm choice
  (LinearSVC over Logistic Regression, reaching macro F1 0.89 as the final model).
- **Error analysis**: identified two distinct failure modes — mixed-sentiment reviews where
  TF-IDF's bag-of-words approach can't weigh a later sentiment shift appropriately, and label
  noise where the review text and the author's star rating don't fully align.

**Key takeaway**: the final macro F1 of 0.89 reflects both the model's real limitations
(no word order awareness) and an inherent ceiling from label noise in the data itself. Sequence-
aware models (e.g. transformer-based architectures) would likely handle the first limitation
better, though the second — inconsistency in human-assigned labels — would remain regardless
of model choice.